# Features Engineering
In this notebook we experiment with some features engineering:

1. using TSFresh

2. and using ROCKET (Random Convolutional Kernel Transform)

NOTE: Before starting exploring this notebook, I recommend checking EDA  notebook first - it contains Exploratory Data Analysis and will help you get some understanding of the datasets.

In [1]:
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime

import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

## First, let's read aircraft engines datasets, we start with "FD001" dataset, which has:

100 engines time series in TRAIN set

100 engines time series in TEST set

1 Fault condition

1 Operating condition

In [2]:
from utils2 import read_dataset, calculate_RUL, SENSOR_COLUMNS

train, test, test_rul = read_dataset("FD001")

train["rul"] = calculate_RUL(train, upper_threshold=125)

print(f"train.shape = {train.shape}")
train.head(200)

train.shape = (20631, 27)


,unit,time_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,rul
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,125
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,125
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,125
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,125
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2,4,0.0035,-0.0004,100.0,518.67,641.68,1584.15,1396.08,14.62,...,2387.93,8140.44,8.4018,0.03,391,2388,100.0,39.13,23.5027,125
196,2,5,0.0005,0.0004,100.0,518.67,641.73,1579.03,1402.52,14.62,...,2387.94,8136.67,8.3867,0.03,390,2388,100.0,39.18,23.4234,125
197,2,6,-0.0010,0.0004,100.0,518.67,641.30,1577.50,1396.76,14.62,...,2387.99,8133.65,8.3800,0.03,392,2388,100.0,39.15,23.4270,125
198,2,7,0.0001,-0.0002,100.0,518.67,642.03,1587.49,1400.65,14.62,...,2388.04,8136.33,8.3941,0.03,391,2388,100.0,39.10,23.4718,125


### Before starting feature engineering - we should reomve sensors with constant values that we saw during  EDA . For that, we create a transformer which drops features with variance lower than a "threshold" .


In [3]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold


class LowVarianceFeaturesRemover(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0):
        self.threshold = threshold
        self.selector = VarianceThreshold(threshold=threshold)

    def fit(self, X):
        self.selector.fit(X)
        return self

    def transform(self, X):
        X_t = self.selector.transform(X)
        droped_features = X.columns[~self.selector.get_support()]
        print(f"Droped low variance features: {droped_features.to_list()}")
        return pd.DataFrame(X_t, columns=self.selector.get_feature_names_out())

In [4]:
from utils2 import SENSOR_COLUMNS


class ScalePerEngine(BaseEstimator, TransformerMixin):
    """
    Scale individual engines time series with respect to its start.
    Substract firts `n_first_cycles` AVG values from time series.
    """

    def __init__(self, n_first_cycles=20, sensors_columns=SENSOR_COLUMNS):
        self.n_first_cycles = n_first_cycles
        self.sensors_columns = sensors_columns

    def fit(self, X):
        return self

    def transform(self, X):
        self.sensors_columns = [x for x in X.columns if x in self.sensors_columns]

        init_sensors_avg = (
            X[X["time_cycles"] <= self.n_first_cycles]
            .groupby(by=["unit"])[self.sensors_columns]
            .mean()
            .reset_index()
        )

        X_t = X[X["time_cycles"] > self.n_first_cycles].merge(
            init_sensors_avg, on=["unit"], how="left", suffixes=("", "_init_v")
        )

        for SENSOR in self.sensors_columns:
            X_t[SENSOR] = X_t[SENSOR] - X_t["{}_init_v".format(SENSOR)]

        drop_columns = X_t.columns.str.endswith("init_v")
        return X_t[X_t.columns[~drop_columns]]

## Feature Engineerinng with TSFRESH


NOTE: 
While TSfresh automates a lot of features engineering - it also tends to extract highly correlated features - see [1]. In order to eliminate this problem, the authors of the paper and the library propose to perform Principal Component Analysis. PCA can be done right after TSfresh generates a candidate features and before the features selection procedure. Or PCA can be performed after TSFresh does features selection. We stick to the first option and apply features selection on principal components.

[1] Christ, M., Kempa-Liehr, A.W. and Feindt, M. (2016). Distributed and parallel time series feature extraction for industrial big data applications. ArXiv e-prints: 1610.07717 URL: http://adsabs.harvard.edu/abs/2016arXiv161007717C




### Our TSFRESH feature engineering pipeline is:

1.transform the Original time series into sliding window of length 30 (roll_time_series method)

2.Calculate TSFresh features from the predifined list of features 

3.Perform Principal Component Analysis (PCA)

4.Do feature selection with TSFresh.

In [5]:
from tsfresh.utilities.dataframe_functions import roll_time_series

In [17]:
class RollTimeSeries(BaseEstimator, TransformerMixin):
    def __init__(self, min_timeshift, max_timeshift, rolling_direction):
        self.min_timeshift = min_timeshift
        self.max_timeshift = max_timeshift
        self.rolling_direction = rolling_direction

    def fit(self, X):
        return self

    def transform(self, X):
        _start = datetime.now()
        print("Start Rolling TS")
        X_t = roll_time_series(
            X,
            column_id="unit",
            column_sort="time_cycles",
            rolling_direction=self.rolling_direction,
            min_timeshift=self.min_timeshift,
            max_timeshift=self.max_timeshift,
        )
        print(f"Done Rolling TS in {datetime.now() - _start}")
        return X_t

### 
NOTE: TSfresh suggests a couple of options to choose a set of features from - e.g. MinimalFCParameters, ComprehensiveFCParameters, EfficientFCParameters - see https://tsfresh.readthedocs.io/en/latest/text/feature_extraction_settings.html.

However, after reviewing the proposed features lists, a set of specific features was shortlisted - see below:

In [18]:
tsfresh_calc = {
    "mean_change": None,
    "mean": None,
    "standard_deviation": None,
    "root_mean_square": None,
    "last_location_of_maximum": None,
    "first_location_of_maximum": None,
    "last_location_of_minimum": None,
    "first_location_of_minimum": None,
    "maximum": None,
    "minimum": None,
    "time_reversal_asymmetry_statistic": [{"lag": 1}, {"lag": 2}, {"lag": 3}],
    "c3": [{"lag": 1}, {"lag": 2}, {"lag": 3}],
    "cid_ce": [{"normalize": True}, {"normalize": False}],
    "autocorrelation": [
        {"lag": 0},
        {"lag": 1},
        {"lag": 2},
        {"lag": 3},
    ],
    "partial_autocorrelation": [
        {"lag": 0},
        {"lag": 1},
        {"lag": 2},
        {"lag": 3},
    ],
    "linear_trend": [{"attr": "intercept"}, {"attr": "slope"}, {"attr": "stderr"}],
    "augmented_dickey_fuller": [
        {"attr": "teststat"},
        {"attr": "pvalue"},
        {"attr": "usedlag"},
    ],
    "linear_trend_timewise": [{"attr": "intercept"}, {"attr": "slope"}],
    "lempel_ziv_complexity": [
        {"bins": 2},
        {"bins": 3},
        {"bins": 5},
        {"bins": 10},
        {"bins": 100},
    ],
    "permutation_entropy": [
        {"tau": 1, "dimension": 3},
        {"tau": 1, "dimension": 4},
        {"tau": 1, "dimension": 5},
        {"tau": 1, "dimension": 6},
        {"tau": 1, "dimension": 7},
    ],
    "fft_coefficient": [
        {"coeff": 0, "attr": "abs"},
        {"coeff": 1, "attr": "abs"},
        {"coeff": 2, "attr": "abs"},
        {"coeff": 3, "attr": "abs"},
        {"coeff": 4, "attr": "abs"},
        {"coeff": 5, "attr": "abs"},
        {"coeff": 6, "attr": "abs"},
        {"coeff": 7, "attr": "abs"},
        {"coeff": 8, "attr": "abs"},
        {"coeff": 9, "attr": "abs"},
        {"coeff": 10, "attr": "abs"},
    ],
    "fft_aggregated": [
        {"aggtype": "centroid"},
        {"aggtype": "variance"},
        {"aggtype": "skew"},
        {"aggtype": "kurtosis"},
    ],
}

In [19]:
from tsfresh import extract_features, select_features
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from utils2 import calculate_RUL, SENSOR_COLUMNS

In [20]:
class TSFreshFeaturesExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, calc=tsfresh_calc):
        self.calc = calc

    def _clean_features(self, X):
        old_shape = X.shape
        X_t = X.T.drop_duplicates().T
        print(f"Droped {old_shape[1] - X_t.shape[1]} duplicate features")

        old_shape = X_t.shape
        X_t = X_t.dropna(axis=1)
        print(f"Droped {old_shape[1] - X_t.shape[1]} features with NA values")
        return X_t

    def fit(self, X):
        return self

    def transform(self, X):
        _start = datetime.now()
        print("Start Extracting Features")
        X_t = extract_features(
            X[
                ["id", "time_cycles"]
                + X.columns[X.columns.str.startswith("sensor")].tolist()
            ],
            column_id="id",
            column_sort="time_cycles",
            default_fc_parameters=self.calc,
        )
        print(f"Done Extracting Features in {datetime.now() - _start}")
        X_t = self._clean_features(X_t)
        return X_t


class CustomPCA(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=None, random_state=None):
        self.n_components = n_components
        self.random_state = random_state

    def fit(self, X):
        assert "unit" not in X.columns, "columns should be only features"
        self.ftr_columns = X.columns

        self.scaler = StandardScaler()
        X_sc = self.scaler.fit_transform(X[self.ftr_columns].values)

        self.pca = PCA(n_components=self.n_components, random_state=self.random_state)
        self.pca.fit_transform(X_sc)
        return self

    def transform(self, X):
        X_sc = self.scaler.transform(X[self.ftr_columns].values)
        X_pca = self.pca.transform(X_sc)
        return pd.DataFrame(X_pca, index=X.index)


class TSFreshFeaturesSelector(BaseEstimator, TransformerMixin):
    def __init__(self, fdr_level=0.001):
        self.fdr_level = fdr_level

    def fit(self, X):
        rul = calculate_RUL(
            X.index.to_frame(name=["unit", "time_cycles"]).reset_index(drop=True),
            upper_threshold=135,
        )

        X_t = select_features(X, rul, fdr_level=self.fdr_level)
        self.selected_ftr = X_t.columns

        print(
            f"Selected {len(self.selected_ftr)} out of {X.shape[1]} features: "
            f"{self.selected_ftr.to_list()}"
        )
        return self

    def transform(self, X):
        return X[self.selected_ftr]

In [21]:
from sklearn.pipeline import Pipeline

tsfresh_pipe = Pipeline(
    [
        # Cleaning
        ("drop-low-variance", LowVarianceFeaturesRemover(threshold=0)),
        # Scaling and Preprocessing
        (
            "scale-per-engine",
            ScalePerEngine(n_first_cycles=15, sensors_columns=SENSOR_COLUMNS),
        ),
        (
            "roll-time-series",
            RollTimeSeries(min_timeshift=29, max_timeshift=29, rolling_direction=1),
        ),
        # TSFresh features engineering
        ("extract-tsfresh-features", TSFreshFeaturesExtractor(calc=tsfresh_calc)),
        ("PCA", CustomPCA(n_components=40)),
        ("features-selection", TSFreshFeaturesSelector(fdr_level=0.001)),
    ]
)

In [11]:
train_tsfresh_ftr = tsfresh_pipe.fit_transform(train)

train_tsfresh_ftr.head(20)

Droped low variance features: ['op_setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Start Rolling TS


Rolling: 100%|██████████| 29/29 [00:10<00:00,  2.71it/s]


Done Rolling TS in 0:00:11.270408
Start Extracting Features


Feature Extraction: 100%|██████████| 30/30 [04:45<00:00,  9.53s/it]


Done Extracting Features in 0:05:18.122399
Droped 19 duplicate features
Droped 14 features with NA values
Selected 11 out of 40 features: [0, 2, 3, 4, 5, 1, 39, 31, 15, 20, 10]


0         2         3         4         5         1   \
1.0 45.0 -11.033204 -0.097374 -6.881436 -2.593459  0.143013  0.106557   
    46.0 -10.807696 -0.277100 -6.757699 -2.676369 -0.073365  0.470533   
    47.0 -11.188236 -0.256071 -7.048327 -3.087324 -0.213330  0.472277   
    48.0 -11.003254 -0.213297 -7.195406 -2.599212 -0.105324  0.477711   
    49.0 -10.618319 -0.169757 -6.951875 -1.833905 -0.364811  0.389854   
    50.0 -10.740795 -0.583869 -6.451398 -1.868283 -0.393531  0.226121   
    51.0 -10.446092 -0.536986 -6.121546 -1.493972 -0.315201  0.260667   
    52.0  -9.729545 -0.622703 -5.940951 -1.248458 -1.242589  0.726448   
    53.0 -10.042239 -0.535461 -5.975891 -1.580959 -0.991202  0.400304   
    54.0  -9.673151 -0.627448 -5.981307 -0.672275 -0.884130 -0.180607   
    55.0 -10.110125 -0.694498 -5.924392 -1.337312 -1.223740  0.101423   
    56.0 -10.092037 -0.542706 -6.277718 -0.685447 -1.102002 -0.268107   
    57.0  -9.901101 -0.707458 -6.305038 -0.501161 -1.327775 -0.166675   
    58.0 -10.101792 -0.608376 -6.446081 -0.515003 -0.563814 -0.487988   
    59.0  -9.345976 -0.705652 -6.369627  0.149937 -0.521634 -0.493176   
    60.0 -10.069884 -0.877701 -6.612859 -1.238307 -0.491494 -0.256181   
    61.0  -9.580533 -0.785076 -6.614148 -0.145690 -0.780857 -0.873527   
    62.0  -9.481169 -0.864668 -6.659817  0.124284 -0.461917 -0.954286   
    63.0  -9.169525 -0.838820 -6.720970  0.293002 -0.373036 -0.513055   
    64.0  -9.090643 -0.741063 -6.787138  0.281213 -1.339965 -0.379266   

                39        31        15        20        10  
1.0 45.0  3.421307  1.629908 -0.406896 -1.880015  1.589181  
    46.0  3.450387  1.396677 -0.388162 -2.685731  1.783030  
    47.0  3.427932  1.351520 -0.232821 -2.819945  1.474932  
    48.0  2.593886  1.393309 -0.045937 -2.356941  1.666134  
    49.0  1.968785  1.373347 -0.188195 -1.873619  1.125662  
    50.0  1.026654  1.516566 -0.006619 -1.805419  0.771191  
    51.0  1.821588  2.291583 -0.447427 -1.969006  0.979132  
    52.0  1.818270  2.461650  0.683459 -2.821395  0.981458  
    53.0  2.462301  2.499705  0.397122 -2.401057 -0.181380  
    54.0  1.755201  1.783717  0.955356 -3.023987 -1.084110  
    55.0  1.601158  1.114514  0.308665 -2.717519 -1.816972  
    56.0  1.523100  1.849326 -0.111918 -2.311575 -3.048810  
    57.0  2.769670  1.762094  0.140368 -3.035782 -3.029629  
    58.0  3.171325  1.265032 -0.059922 -2.535955 -3.712599  
    59.0  3.383406  1.551709  0.570216 -1.540383 -3.041669  
    60.0  3.943383  0.453797  1.290083 -1.286228 -2.932241  
    61.0  4.169694 -0.499904  1.004591 -1.724150 -3.611175  
    62.0  4.888100  0.788664  0.260543 -1.184541 -3.064194  
    63.0  4.160801 -0.812552 -0.448581 -1.418897 -3.388925  
    64.0  4.165555 -1.445659 -0.635911 -1.536080 -3.829869

### Features Engineering with ROCKET
ROCKET (Random Convolutional Kernel Transform) transforms time series into features using random convolutional kernels (by default 10000 kernels) and applying max pooling and proportion of positive values (which produces 2 features per kernel). This results in (by default) 20000 features.

[1] Dempster A., Petitjean F. and Webb G., "ROCKET: Exceptionally fast and accurate time series classification using random convolutional kernels": https://arxiv.org/pdf/1910.13051.pdf

Note: here we add extra step for scaling the time series before running ROCKET - see below:

In [12]:
from sklearn.preprocessing import StandardScaler


class CustomStandardScaler(BaseEstimator, TransformerMixin):
    def fit(self, X):
        self.scale_columns = X.columns[
            X.columns.str.startswith("sensor") | X.columns.str.startswith("op_setting")
        ]
        self.scaler = StandardScaler()
        self.scaler.fit(X[self.scale_columns])
        return self

    def transform(self, X):
        X_t = self.scaler.transform(X[self.scale_columns])
        df = (
            pd.concat(
                [
                    X[[c for c in X.columns if c not in self.scale_columns]],
                    pd.DataFrame(X_t, columns=self.scale_columns),
                ],
                axis=1,
            )
            .sort_values(["unit", "time_cycles"])
            .reset_index(drop=True)
        )
        return df

In [13]:
from datetime import datetime

from sktime.datatypes._panel._convert import from_multi_index_to_nested


class TransformTS2Nested(BaseEstimator, TransformerMixin):
    """
    See https://www.sktime.org/en/stable/examples/loading_data.html#Using-multi-indexed-pandas-DataFrames
    """

    def fit(self, X):
        return self

    def transform(self, X):
        assert "id" in X.columns, "X should have `id` column"
        X_mi = X.copy()
        X_mi.index = pd.MultiIndex.from_frame(X[["id", "time_cycles"]])
        X_mi = X_mi.drop(columns=["rul", "id"], errors="ignore")

        _start = datetime.now()
        print("Start Converting multi-index DF to sktime nested DF")
        X_nested = from_multi_index_to_nested(X_mi, instance_index="id")

        X_nested.index = pd.MultiIndex.from_frame(
            pd.DataFrame(
                {
                    "unit": X_nested["unit"].apply(lambda x: x.unique()[0]),
                    "time_cycles": X_nested["time_cycles"].apply(lambda x: x.iloc[-1]),
                }
            )
        )
        X_nested = X_nested.drop(columns=["unit", "time_cycles"])
        print(
            f"Converted multi-index DF to sktime nested DF in {datetime.now() - _start}"
        )

        return X_nested

In [14]:
from sktime.transformations.panel.rocket import Rocket


class RocketTransform(BaseEstimator, TransformerMixin):
    """
    See https://www.sktime.org/en/stable/examples/rocket.html
    """

    def __init__(
        self, num_kernels=10000, normalise=False, n_jobs=-1, random_state=None
    ):
        self.num_kernels = num_kernels
        self.normalise = normalise
        self.n_jobs = n_jobs
        self.random_state = random_state

    def fit(self, X):
        self.rocket = Rocket(
            num_kernels=self.num_kernels,
            normalise=self.normalise,
            n_jobs=self.n_jobs,
            random_state=self.random_state,
        )
        self.rocket.fit(X)
        return self

    def transform(self, X):
        rocket_t = self.rocket.transform(X)
        rocket_t.index = X.index
        return rocket_t

In [15]:
from sklearn.pipeline import Pipeline

from utils2 import SENSOR_COLUMNS


rocket_pipe = Pipeline(
    [
        # Cleaning constant features
        ("drop-low-variance", LowVarianceFeaturesRemover()),
        # Scaling sensors values
        (
            "scale-per-engine",
            ScalePerEngine(n_first_cycles=15, sensors_columns=SENSOR_COLUMNS),
        ),
        ("scale", CustomStandardScaler()),
        # Preprocessing for ROCKET
        (
            "roll-time-series",
            RollTimeSeries(min_timeshift=29, max_timeshift=29, rolling_direction=1),
        ),
        ("nest-time-series", TransformTS2Nested()),
        # Features Engineering w ROCKET
        (
            "rocket",
            RocketTransform(num_kernels=10000, normalise=False, random_state=2021),
        ),
    ]
)

In [16]:
train_rocket_ftr = rocket_pipe.fit_transform(train)

train_rocket_ftr.shape

Droped low variance features: ['op_setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Start Rolling TS


Rolling: 100%|██████████| 29/29 [00:09<00:00,  3.20it/s]


Done Rolling TS in 0:00:09.582590
Start Converting multi-index DF to sktime nested DF
Converted multi-index DF to sktime nested DF in 0:01:09.409110


(16231, 20000)